# Experiment 1 — Matched-scale control

**This is the experiment that decides whether the paper is Q1.**

Branch A collapses from 0.898 to 0.238 macro-F1 under compound shift. It was
trained on 196,854 rows; the published detector it is compared against was
trained on roughly 847,000. Two explanations fit that, and the paper currently
cannot separate them:

1. the detector is sensitive to genuinely withheld categories, or
2. it was simply trained on less data.

This notebook trains the same model, with the same categories withheld, on all
three DroidCollection training shards instead of one. If it still collapses,
explanation 2 is excluded.

## Settings in the right-hand panel

| Setting | Value |
|---|---|
| **Accelerator** | `GPU T4 x2` |
| **Internet** | `On` |
| **Persistence** | `Files only` |

Then **Save Version -> Save & Run All (Commit)**. Do not use interactive Run
All: a browser disconnect kills an interactive session, while a committed run
continues on Kaggle's servers whether or not your laptop is awake.

**Budget: 8-10 hours.** Kaggle's session cap is 12 h.

In [ ]:
import os, sys, time, subprocess, shutil, pathlib, json

T0 = time.time()
def elapsed(label=""):
    m = (time.time() - T0) / 60
    print(f"[{m:6.1f} min] {label}", flush=True)

def run(cmd):
    """Run a pipeline stage and stop the notebook if it fails.

    Without the raise a failed stage prints a traceback and the next cell
    happily trains on whatever stale data is lying around.
    """
    print(">>", " ".join(str(c) for c in cmd), flush=True)
    r = subprocess.run([sys.executable, "-u", *[str(c) for c in cmd]])
    if r.returncode != 0:
        raise SystemExit(f"FAILED: {' '.join(str(c) for c in cmd)}")

print("Python", sys.version.split()[0])
import torch
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f"  GPU{i}: {p.name}  {p.total_memory/1e9:.1f} GB")
else:
    raise SystemExit("No GPU. Set Accelerator to GPU T4 x2 in the right panel.")

## 1. Code

In [ ]:
WORK = pathlib.Path("/kaggle/working/project")
WORK.mkdir(parents=True, exist_ok=True)

def find_aicd():
    root = pathlib.Path("/kaggle/input")
    if not root.exists():
        return None
    for cand in root.rglob("aicd"):
        if (cand / "config.py").exists() and (cand / "models").is_dir():
            return cand
    return None

src = find_aicd()
if src is None:
    raise SystemExit(
        "aicd/ not found under /kaggle/input.\n"
        "Add your code dataset: right panel -> Input -> Add Input -> Datasets,\n"
        "then search for the dataset you created from aicd-code.zip.")

dest = WORK / "aicd"
if dest.exists():
    shutil.rmtree(dest)
shutil.copytree(src, dest)
os.chdir(WORK)
sys.path.insert(0, str(WORK))
print("code ->", dest)
print("configs present:", sorted(p.name for p in (dest/"configs").glob("kaggle*.yaml")))

## 2. Dependencies

In [ ]:
pkgs = ["xgboost", "tree-sitter", "tree-sitter-language-pack",
        "datasets", "shap", "pyyaml", "scikit-learn", "pyarrow"]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=False)
import importlib
for m in ["xgboost", "sklearn", "transformers", "datasets", "yaml"]:
    mod = importlib.import_module(m)
    print(f"  ok  {m:14s} {getattr(mod, '__version__', '')}")
elapsed("deps")

## 3. Build the corpus from all three training shards

`--train-shards 3` is the only change from the original run. Downloading and
filtering roughly 847k rows takes about an hour.

The split-integrity tests run straight after and **must pass**. If a withheld
category leaks into training, every number downstream is fiction, and the whole
point of this experiment is that the withholding is real.

In [ ]:
CFG = "kaggle_matched.yaml"

run(["-m", "aicd.data.download", "--config", CFG, "--train-shards", "3"])
elapsed("downloaded 3 shards")

for stage in ["normalize", "filter", "splits"]:
    run(["-m", f"aicd.data.{stage}", "--config", CFG])
    elapsed(stage)

run(["-m", "pytest", "aicd/tests/", "-q"])
elapsed("split-integrity tests passed")

## 4. Confirm the training set actually grew

The experiment is meaningless if this number is not much larger than the
196,854 rows of the original run. Check it before spending seven hours of GPU
on it.

In [ ]:
import pandas as pd
sp = json.load(open(WORK / "aicd" / "eval" / "reports" / "splits.json"))
rows = sp["train"]["rows"]
print(f"training rows: {rows:,}   (original run: 196,854)")
print(f"ratio: {rows/196854:.2f}x")
if rows < 300000:
    raise SystemExit(
        f"Only {rows:,} training rows. Expected roughly 545,000.\\n"
        "The extra shards did not make it into the split. Check that cell 3\\n"
        "passed --train-shards 3 and that download.py did not skip cached\\n"
        "single-shard files; delete artifacts/data/hf/ and re-run if so.")
print("\\nper-class:", sp["train"]["labels"])

## 5. Train

`--tag matched` keeps this run's checkpoint, weights, report and probability
arrays separate from the original run's. Without a distinct tag the two
overwrite each other, and `--resume` would continue training the *other*
model.

`--resume` makes the cell safe to re-run: each epoch writes a checkpoint with
model, optimiser, scheduler and scaler state, so an interrupted run picks up at
the next epoch rather than starting over.

Roughly 6-8 hours for three epochs at this scale. If the session is killed,
re-run this one cell.

In [ ]:
run(["-m", "aicd.models.modernbert_triplet", "--config", CFG,
     "--tag", "matched", "--resume"])
elapsed("branch A (matched scale) trained and evaluated")

## 6. Read the result

In [ ]:
rep = WORK / "aicd" / "eval" / "reports" / "branch_a_matched.json"
r = json.load(open(rep))["slices"]
print(f"{'condition':24s} {'matched':>9s} {'original':>9s}")
print("-" * 45)
ORIG = {"s1_in_distribution": 0.8977, "s2_unseen_generator": 0.8685,
        "s3_unseen_language": 0.5667, "s4_unseen_domain": 0.4029,
        "s5_compound": 0.2378}
for s, o in ORIG.items():
    if s in r:
        print(f"{s:24s} {r[s]['macro_f1']:9.4f} {o:9.4f}")
s1 = r["s1_in_distribution"]["macro_f1"]; s5 = r["s5_compound"]["macro_f1"]
print(f"\ncollapse S1 -> S5: {s1:.4f} -> {s5:.4f}  (drop {s1-s5:.4f})")
print(f"original run drop: {0.8977-0.2378:.4f}")
print()
if s5 < 0.45:
    print("COLLAPSE PERSISTS at roughly 3x the training data.")
    print("Data volume is excluded. The paper's causal claim holds.")
else:
    print("COLLAPSE DOES NOT PERSIST. This is a real finding and must be")
    print("reported: part of the original effect was a data-volume artefact,")
    print("and the paper's framing has to change to match.")

## 7. Save

In [ ]:
OUT = pathlib.Path("/kaggle/working/results")
OUT.mkdir(parents=True, exist_ok=True)

reports = WORK / "aicd" / "eval" / "reports"
if reports.exists():
    shutil.copytree(reports, OUT / "reports", dirs_exist_ok=True)

# The probability arrays are what the analysis modules re-read at home, and
# they are small. The model weights are hundreds of MB and are not needed to
# reproduce any number in the paper, so they stay behind.
art = WORK / "aicd" / "artifacts"
npy = OUT / "arrays"
npy.mkdir(exist_ok=True)
n = 0
for f in art.glob("proba_a*.npy"):
    shutil.copy(f, npy / f.name); n += 1
for f in art.glob("labels.parquet"):
    shutil.copy(f, npy / f.name)
if (art / "kaggle").exists():
    for f in (art / "kaggle").glob("*"):
        if f.is_file() and f.stat().st_size < 200e6:
            shutil.copy(f, npy / f.name); n += 1

shutil.make_archive("/kaggle/working/results", "zip", OUT)
print(f"copied {n} arrays")
print("-> /kaggle/working/results.zip  (download this from the Output tab)")
for p in sorted(OUT.rglob("*")):
    if p.is_file():
        print(f"  {p.stat().st_size/1024:8.0f} KB  {p.relative_to(OUT)}")
elapsed("saved")